### 전처리된 모든 데이터를 벡터료 변경하여 S3에 저장하는 노트북 입니다.

### 이미지 벡터화

In [47]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error
import joblib

import boto3


import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tcn import TCN

In [48]:
s3 = boto3.client('s3', region_name='ap-northeast-2')

bucket_name = 'real-estate-avm'

yester_day = '2025-09-24'

prefix = f'processed/normalized-images/dt={yester_day}/'
local_dir = 'data/'

response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

for obj in response.get('Contents', []):
    key = obj['Key']
    # .jpg가 없으면 확장자 없이 그냥 다운로드함...
    filename = key.split('/')[-1]+'.jpg'
    s3.download_file(bucket_name, key, os.path.join(local_dir, filename))

In [49]:
BATCH_SIZE = 64
IMAGE_PCA_COMPONENTS = 32

def extract_image_features_2048d(image_paths, batch_size=BATCH_SIZE):
    """ResNet50으로 고차원 특징 벡터를 추출하는 헬퍼 함수"""
    image_input = Input(shape=(224, 224, 3), name='image_input')
    base_cnn = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_tensor=image_input)
    base_cnn.trainable = False
    extractor_model = Model(inputs=image_input, outputs=base_cnn.output, name='resnet50_feature_extractor')
    
    def image_generator(paths, b_size):
        for i in range(0, len(paths), b_size):
            batch_paths = paths[i:i+b_size]
            batch_images = []
            for p in batch_paths:
                try:
                    img = load_img(p, target_size=(224, 224))
                    img_array = img_to_array(img)
                    batch_images.append(img_array)
                except (FileNotFoundError, IOError):
                    batch_images.append(np.zeros((224, 224, 3)))
            yield preprocess_input(np.array(batch_images))
            
    num_batches = int(np.ceil(len(image_paths) / batch_size))
    features = extractor_model.predict(image_generator(image_paths, batch_size), steps=num_batches, verbose=1)
    return features

In [50]:

# .jpg 파일 전체 경로 불러오기
image_paths = [
    os.path.join(local_dir, f)
    for f in os.listdir(local_dir)
    if os.path.isfile(os.path.join(local_dir, f)) and f.lower().endswith(".jpg")
]

In [51]:
image_features_2048d = extract_image_features_2048d(image_paths)

1/1 [==============================] - 1s 533ms/step


In [52]:
# 2048차원 이미지 특징을 원하는 크기로 축소
image_input = Input(shape=(2048,), name="image_feature_input")
image_embedding = Dense(IMAGE_PCA_COMPONENTS, activation="relu")(image_input)  # PCA 대신 MLP
image_extractor = Model(inputs=image_input, outputs=image_embedding, name="mlp_image_feature_extractor")


# 이미지 특징 벡터 변환
vector_32d = image_extractor.predict(image_features_2048d)  # train/test 구분 없이 사용 가능

print(vector_32d.shape)

1/1 [==============================] - 0s 29ms/step
(1, 32)


In [53]:
np.save(f'{local_dir}/dt={yester_day}.npy',vector_32d)